In [11]:
from langgraph.graph import StateGraph ,START,END 

from langchain_groq import ChatGroq
from dotenv import load_dotenv 
load_dotenv() 
from typing import TypedDict,Annotated 
from langchain_core.messages import HumanMessage,BaseMessage

from langgraph.checkpoint.memory import MemorySaver  




In [12]:
llm=ChatGroq(model='Llama-3.3-70b-Versatile') 

In [13]:
# llm.invoke('hi') for checking the llm api 


In [14]:
from langgraph.graph.message import add_messages
#here BaseMessage is abstract it can be any kind of message human,system,ai,etc 
# add_messages is a reducer by which we append the messages 

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [15]:
#lets make a obejct for the persistence memory 
checkpointer=MemorySaver() 

#lets make our graph 
graph=StateGraph(ChatState) 


In [16]:
#logic for the chatting 
def Chat_with_LLM(state: ChatState)-> ChatState: 
    response=llm.invoke(state['messages']) 
    return {'messages':[response]}

In [17]:
#lets add node thier is only 1 node in it 
#by which user can communicate with the llm 
graph.add_node('Chat_Node',Chat_with_LLM) 




#lets add edges 
graph.add_edge(START,'Chat_Node') 
graph.add_edge('Chat_Node',END)




In [18]:
chat_bot=graph.compile(checkpointer=checkpointer) 


In [19]:
msg={'messages':HumanMessage(content='Hey make the outline for a blog about llm.')}
# final_output=chat_bot.invoke(msg) 



In [20]:
# ai_response=final_output['messages'][-1].content

In [21]:
from IPython.display import display,Markdown 
# display(Markdown(ai_response))


In [23]:
#lets provide the ability like a chatbot 
while True: 
    user_msg=input('Enter your message: ') 
    print(f'User: {user_msg}') 
    
    if user_msg.strip().lower() in ['exit','quit','stop']: 
        print('Exiting the chat') 
        
        break 
    
    # response=chat_bot.invoke({'messages':[HumanMessage(content=user_msg)]})
    # ai_response=response['messages'][-1].content 
    
    # print(f'AI: {ai_response}') 
    #lets print the proper message using the Ipython 
    # display(Markdown(f'**AI:** {ai_response}'))
    
    #here in the above logic our chatbot have no memory so lets add history 
    #because we are invoking this in every invocation 
    

User: 
User: exit
Exiting the chat


In [ ]:
#lets see Persistence of the chatbot 
#here in the above logic we are always giving new state as input 
#but now we store that previous state and use it in the next invocation
# from langgraph.checkpoint.memory import MemorySaver 

#at the time of invocation we have to define the thread 
thread_id='1'
while True: 
    user_msg=input('Enter your message: ') 
    print(f'User: {user_msg}') 
    
    if user_msg.strip().lower() in ['exit','quit','stop']: 
        print('Exiting the chat') 
        
        break 
    config={'configurable':{'thread_id':thread_id}}
    
    response=chat_bot.invoke({'messages':[HumanMessage(content=user_msg)]},config=config)
    ai_response=response['messages'][-1].content 
    
    # print(f'AI: {ai_response}') 
    #lets print the proper message using the Ipython 
    display(Markdown(f'**AI:** {ai_response}'))
    
    #here in the above logic our chatbot have no memory so lets add history 
    #because we are invoking this in every invocation 
    

User: my name is aashish


**AI:** Hello Aashish! It's nice to meet you. Is there something I can help you with or would you like to chat?

User: can you tell me my name


**AI:** Your name is Aashish.

User: thanks


**AI:** You're welcome, Aashish. Have a great day!

User: exit
Exiting the chat
